In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
def parse_dispatchregionsum(path):
    raw = spark.read.text(path)

    header = (
        raw
        .filter(F.col("value").startswith("I,DISPATCH,REGIONSUM"))
        .select(F.split("value", ",").alias("fields"))
        .first()["fields"]
    )

    time_i = header.index("SETTLEMENTDATE")
    runno_i = header.index("RUNNO")
    region_i = header.index("REGIONID")
    intervention_i = header.index("INTERVENTION")
    demand_i = header.index("TOTALDEMAND")

    df = (
        raw
        .filter(F.col("value").startswith("D,DISPATCH,REGIONSUM"))
        .withColumn("fields", F.split("value", ","))
        .filter(F.col("fields")[region_i] == "VIC1")
        .filter(F.col("fields")[runno_i].cast("int") == 1)
        .filter(F.col("fields")[intervention_i].cast("int") == 0)
        .select(
            F.to_timestamp(
                F.regexp_replace(
                    F.col("fields")[time_i],
                    '"',
                    ''
                ),
                "yyyy/MM/dd HH:mm:ss"
            ).alias("time"),

            F.col("fields")[demand_i]
            .cast("double")
            .alias("demand")
        )
    )

    return df

In [0]:
base_path = "/Volumes/workspace/default/aemo_mlops_volume/bronze"

monthly = parse_dispatchregionsum(
    f"{base_path}/monthly_uncompressed/*.CSV"
)

daily = parse_dispatchregionsum(
    f"{base_path}/daily_uncompressed/*.CSV"
)

current = parse_dispatchregionsum(
    f"{base_path}/current_uncompressed/*.CSV"
)

In [0]:
monthly.createOrReplaceTempView("monthly")
daily.createOrReplaceTempView("daily")
current.createOrReplaceTempView("current")

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.default.demand_vic_5min
USING DELTA
AS

WITH all_data AS (

    SELECT time, demand, 3 AS priority
    FROM monthly

    UNION ALL

    SELECT time, demand, 2 AS priority
    FROM daily

    UNION ALL

    SELECT time, demand, 1 AS priority
    FROM current
),

deduplicated AS (

    SELECT
        time,
        demand,
        ROW_NUMBER() OVER (
            PARTITION BY time
            ORDER BY priority DESC
        ) AS rn

    FROM all_data
)

SELECT
    time,
    demand

FROM deduplicated

WHERE rn = 1;

In [0]:
%sql

SELECT *
FROM workspace.default.demand_vic_5min
ORDER BY time;